# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing data entities by their `@id` for clarity and reproducibility.

### Dataset Source
The dataset source is provided via the FAIR² Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and fields, referencing their `@id`. This is essential for robust data access and referencing in all subsequent steps.

First, let's print all record set `@id`s in this dataset, and for each, print out the available field (column) `@id`s and field names.

In [ ]:
# List all record set @id's and their fields
record_sets = dataset.metadata.recordSets
record_set_ids = [rs['@id'] for rs in record_sets]
print("Record sets found:")
for rs in record_sets:
    print(f"- {rs['@id']} : {rs.get('name', '')}")
    fields = rs.get('fields', [])
    if fields:
        print("    Fields/Columns:")
        for f in fields:
            fname = f.get('name', '')
            fid = f['@id']
            print(f"      - {fid} : {fname}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

For demonstration, we'll extract the main protein expression data record set, which has `@id` = `dv:proteinExpressionMainTable`, and display the fields (columns).

In [ ]:
# Define the main record set(s) by their @id
main_record_set_id = 'dv:proteinExpressionMainTable'  # Change if your record set @id differs
all_record_set_ids = record_set_ids  # In case there are multiple, you can select more

dataframes = {}
for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

if main_record_set_id in dataframes:
    print(f"Fields in {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print(f"No data frame available for {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All field references are by their `@id`.

For this example, let's filter the data based on `cr:coveragePercent` (protein coverage percentage > 10), normalize it, and group by `cr:sample` if that field exists.

In [ ]:
# Replace with the actual numeric field @id for protein coverage. E.g., 'cr:coveragePercent'
numeric_field_id = 'cr:coveragePercent'  # Coverage percentage field @id

df = dataframes.get(main_record_set_id)
if df is not None and numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by sample or another meaningful field by @id
    group_field_id = 'cr:sample'  # e.g., group by sample (change if your dataset uses a different field @id)
    if group_field_id in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(group_field_id)[numeric_field_id]
            .mean()
            .reset_index(name=f"mean_{numeric_field_id}")
        )
        print(f"Grouped by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df.head())
else:
    print(f"Field '{numeric_field_id}' not found in {main_record_set_id}.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset, referencing fields by their `@id`. Here, we plot a histogram for `cr:coveragePercent` and sample-wise means if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Coverage Percentage Distribution
if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title('Distribution of Protein Coverage (%)')
    plt.xlabel('Protein Coverage Percentage (@id: cr:coveragePercent)')
    plt.ylabel('Count')
    plt.show()
    
    # If grouping by 'cr:sample' made sense
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10, 4))
        sns.barplot(data=grouped_df, x=group_field_id, y=f"mean_{numeric_field_id}")
        plt.title('Mean Protein Coverage by Sample')
        plt.xlabel('Sample (@id: cr:sample)')
        plt.ylabel('Mean Coverage (%)')
        plt.xticks(rotation=90)
        plt.show()
else:
    print(f"Cannot plot - field '{numeric_field_id}' missing.")

## 6. Conclusion
In this notebook, we have:
- Loaded and reviewed the Mass Spectrometry dataset using `mlcroissant`
- Explored all record sets and referenced fields by their `@id`
- Loaded protein records and performed simple EDA including normalization and grouping
- Visualized the distribution of key numeric values using proper `@id` references

All operations were done referencing Croissant entities by their `@id` for clear and reproducible data science workflows.